In [1]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

load_dotenv()
print("✅ Imports OK")

c:\Users\asehp\Desktop\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


In [2]:
import pdfplumber

PDF_PATH = "../data/AN.pdf"
texte_complet = ""

with pdfplumber.open(PDF_PATH) as pdf:
    print(f"📄 Nombre de pages : {len(pdf.pages)}")
    
    for page_num, page in enumerate(pdf.pages):
        texte_page = ""
        
        # 1. Extraire les tableaux en premier
        tables = page.extract_tables()
        zones_tableaux = []
        
        for table in tables:
            if not table:
                continue
            # Convertir le tableau en texte lisible
            lignes_tableau = []
            for row in table:
                # Nettoyer les cellules vides
                cellules = [str(cell).strip() if cell else "" for cell in row]
                lignes_tableau.append(" | ".join(cellules))
            texte_table = "\n".join(lignes_tableau)
            texte_page += f"\n[TABLEAU]\n{texte_table}\n[/TABLEAU]\n"
        
        # 2. Extraire le texte normal
        texte_brut = page.extract_text()
        if texte_brut:
            texte_page += texte_brut
        
        if texte_page.strip():
            texte_complet += texte_page + "\n\n"
            print(f"  ✅ Page {page_num+1} : {len(texte_page)} car. | {len(tables)} tableau(x)")
        else:
            print(f"  ⚠️  Page {page_num+1} : vide ou image")

print(f"\n📊 Total extrait : {len(texte_complet)} caractères")

📄 Nombre de pages : 144
  ⚠️  Page 1 : vide ou image
  ✅ Page 2 : 1172 car. | 0 tableau(x)
  ✅ Page 3 : 3209 car. | 0 tableau(x)
  ✅ Page 4 : 2379 car. | 0 tableau(x)
  ✅ Page 5 : 1766 car. | 0 tableau(x)
  ✅ Page 6 : 792 car. | 0 tableau(x)
  ✅ Page 7 : 4469 car. | 2 tableau(x)
  ✅ Page 8 : 4497 car. | 2 tableau(x)
  ✅ Page 9 : 4502 car. | 2 tableau(x)
  ✅ Page 10 : 3941 car. | 2 tableau(x)
  ✅ Page 11 : 4259 car. | 2 tableau(x)
  ✅ Page 12 : 3921 car. | 2 tableau(x)
  ✅ Page 13 : 2863 car. | 1 tableau(x)
  ✅ Page 14 : 4503 car. | 2 tableau(x)
  ✅ Page 15 : 4509 car. | 2 tableau(x)
  ✅ Page 16 : 2429 car. | 2 tableau(x)
  ✅ Page 17 : 4835 car. | 2 tableau(x)
  ✅ Page 18 : 4426 car. | 2 tableau(x)
  ✅ Page 19 : 2746 car. | 1 tableau(x)
  ✅ Page 20 : 359 car. | 0 tableau(x)
  ✅ Page 21 : 155 car. | 0 tableau(x)
  ✅ Page 22 : 3603 car. | 1 tableau(x)
  ✅ Page 23 : 3731 car. | 2 tableau(x)
  ✅ Page 24 : 3547 car. | 1 tableau(x)
  ✅ Page 25 : 3456 car. | 1 tableau(x)
  ✅ Page 26 : 3937 car

In [3]:
def decouper_intelligent(texte, taille=700, chevauchement=100):
    """
    Découpe en respectant :
    - les tableaux (ne pas les couper en deux)
    - les paragraphes (couper sur les sauts de ligne)
    """
    chunks = []
    
    # Séparer d'abord les blocs tableaux du reste
    import re
    blocs = re.split(r'(\[TABLEAU\].*?\[/TABLEAU\])', texte, flags=re.DOTALL)
    
    buffer = ""
    
    for bloc in blocs:
        # Si c'est un tableau → chunk indépendant, ne pas couper
        if bloc.startswith("[TABLEAU]"):
            if buffer.strip():
                chunks.append(buffer.strip())
                buffer = ""
            if len(bloc.strip()) > 50:
                chunks.append(bloc.strip())
        else:
            # Texte normal : découper sur les paragraphes
            paragraphes = bloc.split('\n')
            for para in paragraphes:
                if len(buffer) + len(para) < taille:
                    buffer += para + "\n"
                else:
                    if buffer.strip():
                        chunks.append(buffer.strip())
                    # Chevauchement : garder la fin du buffer précédent
                    mots = buffer.split()
                    overlap = " ".join(mots[-30:]) if len(mots) > 30 else ""
                    buffer = overlap + "\n" + para + "\n"
    
    if buffer.strip():
        chunks.append(buffer.strip())
    
    # Filtrer les chunks trop courts
    chunks = [c for c in chunks if len(c) > 80]
    return chunks

chunks = decouper_intelligent(texte_complet, taille=800, chevauchement=100)
print(f"✅ {len(chunks)} chunks créés")

# Vérification rapide
tableaux_dans_chunks = [c for c in chunks if "[TABLEAU]" in c]
print(f"📊 Dont {len(tableaux_dans_chunks)} chunks contenant un tableau")
print(f"📐 Taille moyenne : {sum(len(c) for c in chunks)//len(chunks)} caractères")
print(f"\n--- Exemple d'un chunk tableau ---")
if tableaux_dans_chunks:
    print(tableaux_dans_chunks[0][:400])

chunks = decouper_intelligent(texte_complet, taille=800, chevauchement=100)

print(f"✅ Nombre de chunks créés : {len(chunks)}")

if len(chunks) > 0:
    print(f"\n--- Exemple chunk #1 ---\n{chunks[0]}")

if len(chunks) > 4:
    print(f"\n--- Exemple chunk #5 ---\n{chunks[4]}")

✅ 626 chunks créés
📊 Dont 32 chunks contenant un tableau
📐 Taille moyenne : 772 caractères

--- Exemple d'un chunk tableau ---
[TABLEAU]
Licence en Arts Numériques - 1ÈRE ANNÉE |  |  |  |  |  |  | 
Semestre 1 |  |  |  |  |  |  | 
Semestre
Module | Code l’UE | Intitulé UE | CM
(H) | TD
(H) | TP
(H) | Total
(H) | Total
Crédits
ANL-1 | ANL-1011 | Bases du raisonnement logique et
Bureautique | 10 | 20 | 0 | 30 | 2
 | ANL-1021 | Introduction aux Sciences de
l’Ingénieur | 10 | 20 | 0 | 30 | 2
 | ANL-1031 | Notions de Géométrie 
✅ Nombre de chunks créés : 626

--- Exemple chunk #1 ---
UNIVERSITÉ DE YAOUNDÉ I
École Nationale Supérieure Polytechnique de Yaoundé
PROGRAMME DE FORMATION DE LA FILIÈRE
ARTS NUMÉRIQUES
Spécialité DESIGN / STYLISME NUMÉRIQUE
Design graphique, d’Interface & de Packaging
Design de Produit & Stylisme numérique
Design d’Espace et des couleurs
Webdesign
L3+M1+M2+D
Sciences de l’Ingénieur
Spécialité « CINÉMA D’ANIMATION »
ARTS NUMÉRIQUES
Animation 2D et 3D
LMD en Sciences d

In [4]:
# ---- Cellule diagnostic chunking ----

print(f"{'='*50}")
print(f"  RAPPORT DE CHUNKING")
print(f"{'='*50}")
print(f"Nombre total de chunks    : {len(chunks)}")
print(f"Taille moyenne d'un chunk : {sum(len(c) for c in chunks) // len(chunks)} caractères")
print(f"Chunk le plus court       : {min(len(c) for c in chunks)} caractères")
print(f"Chunk le plus long        : {max(len(c) for c in chunks)} caractères")
print(f"{'='*50}\n")

# Vérifier que les chunks ne sont pas vides ou trop courts
chunks_trop_courts = [c for c in chunks if len(c) < 100]
chunks_ok          = [c for c in chunks if len(c) >= 100]

print(f"✅ Chunks exploitables (≥100 car.)  : {len(chunks_ok)}")
print(f"⚠️  Chunks trop courts (<100 car.)  : {len(chunks_trop_courts)}")

# Vérifier le chevauchement : le début du chunk N+1 doit rappeler la fin du chunk N
print(f"\n--- Vérification du chevauchement ---")
fin_chunk_1   = chunks[0][-60:]
debut_chunk_2 = chunks[1][:60:]
print(f"Fin chunk #1   : ...{fin_chunk_1}")
print(f"Début chunk #2 : {debut_chunk_2}...")

# Afficher 3 chunks aléatoires pour contrôle visuel
import random
print(f"\n--- 3 chunks aléatoires ---")
for i, idx in enumerate(random.sample(range(len(chunks)), 3)):
    print(f"\n[Chunk #{idx}] ({len(chunks[idx])} car.)")
    print(chunks[idx][:200] + "...")

  RAPPORT DE CHUNKING
Nombre total de chunks    : 626
Taille moyenne d'un chunk : 772 caractères
Chunk le plus court       : 145 caractères
Chunk le plus long        : 2086 caractères

✅ Chunks exploitables (≥100 car.)  : 626
⚠️  Chunks trop courts (<100 car.)  : 0

--- Vérification du chevauchement ---
Fin chunk #1   : ...de Génie informatique
(GI)
ARTS NUMÉRIQUES
Titre d’Ingénieur
Début chunk #2 : jeux vidéo L3+M1+M2+D en Sciences de l’Ingénieur Spécialité ...

--- 3 chunks aléatoires ---

[Chunk #451] (793 car.)
artistique, narrative et technique des jeux vidéo, et doit couvrir les aspects suivants : -La redéfinition du Game Design du point de vue de la sémiotique et de la ludologie.
-La pratique de la sémiot...

[Chunk #236] (789 car.)
nouveaux au service d’un concept artistique (moteurs, algorithmes et autres). -Méthodes pour évaluer le potentiel commercial d’un Projet ou d’une Solution, de prototypage, de Versioning et de mise en ...

[Chunk #473] (792 car.)
monde. NB : Un Projet

In [5]:
# Modèle multilingue performant, fonctionne très bien en français
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"⏳ Chargement du modèle : {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("✅ Modèle chargé !")

# Test rapide
test = model.encode(["Bonjour, voici un test d'embedding en français."])
print(f"📐 Dimension des vecteurs : {test.shape[1]}")

⏳ Chargement du modèle : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3890.13it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé !
📐 Dimension des vecteurs : 384


In [6]:
# Initialiser ChromaDB (stockage local dans le dossier chroma_db)
import os
CHROMA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "chroma_db")
print(f"📂 Sauvegarde dans : {CHROMA_PATH}")

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Supprimer la collection si elle existe déjà (pour repartir propre)
try:
    client.delete_collection("an_enspy")
    print("🗑️  Ancienne collection supprimée")
except:
    pass

# Créer la collection
collection = client.create_collection(
    name="an_enspy",
    metadata={"hnsw:space": "cosine"}
)
print("✅ Collection ChromaDB créée : an_enspy")

# Vectoriser et insérer les chunks
print(f"\n⏳ Vectorisation de {len(chunks)} chunks...")

BATCH_SIZE = 50  # Traiter par lots pour éviter les problèmes de mémoire

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i+BATCH_SIZE]
    
    # Créer les embeddings
    embeddings = model.encode(batch).tolist()
    
    # Créer les IDs et métadonnées
    ids = [f"chunk_{j}" for j in range(i, i+len(batch))]
    metadatas = [{"source": "AN.pdf", "chunk_index": j} for j in range(i, i+len(batch))]
    
    # Insérer dans ChromaDB
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )
    
    print(f"  ✅ Lot {i//BATCH_SIZE + 1} inséré ({len(batch)} chunks)")

print(f"\n🎉 Ingestion terminée ! {collection.count()} chunks stockés dans ChromaDB.")

📂 Sauvegarde dans : c:\Users\asehp\Desktop\RAG\notebooks\chroma_db
🗑️  Ancienne collection supprimée
✅ Collection ChromaDB créée : an_enspy

⏳ Vectorisation de 626 chunks...
  ✅ Lot 1 inséré (50 chunks)
  ✅ Lot 2 inséré (50 chunks)
  ✅ Lot 3 inséré (50 chunks)
  ✅ Lot 4 inséré (50 chunks)
  ✅ Lot 5 inséré (50 chunks)
  ✅ Lot 6 inséré (50 chunks)
  ✅ Lot 7 inséré (50 chunks)
  ✅ Lot 8 inséré (50 chunks)
  ✅ Lot 9 inséré (50 chunks)
  ✅ Lot 10 inséré (50 chunks)
  ✅ Lot 11 inséré (50 chunks)
  ✅ Lot 12 inséré (50 chunks)
  ✅ Lot 13 inséré (26 chunks)

🎉 Ingestion terminée ! 626 chunks stockés dans ChromaDB.


In [7]:
# Tester que la recherche fonctionne
QUESTION_TEST = " qu'est ce que la filière Art Numérique ?"

# Vectoriser la question
query_embedding = model.encode([QUESTION_TEST]).tolist()

# Chercher les 3 chunks les plus proches
resultats = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"🔍 Question : {QUESTION_TEST}\n")
print("📌 Chunks les plus pertinents trouvés :\n")

for i, (doc, distance) in enumerate(zip(
    resultats["documents"][0],
    resultats["distances"][0]
)):
    print(f"--- Résultat {i+1} (score : {1-distance:.3f}) ---")
    print(doc[:300])
    print()
    
# trier manuellement (optionnel)
docs = list(zip(resultats["documents"][0], resultats["distances"][0]))
docs_sorted = sorted(docs, key=lambda x: x[1])

for i, (doc, distance) in enumerate(docs_sorted):
    print(f"{i+1}. {1-distance:.4f}")    
    

🔍 Question :  qu'est ce que la filière Art Numérique ?

📌 Chunks les plus pertinents trouvés :

--- Résultat 1 (score : 0.667) ---
: comment l’Art conceptuel nourrit les formes d’Art créées à partir de l’Ordinateur de nos jours : peinture digitale, conception de monuments assistée par ordinateur, design d’intérieurs, jeux vidéo, installations
immersives, Veejaying, etc.
-Un approfondissement du lien entre Art et Science (Medial

--- Résultat 2 (score : 0.657) ---
couvrir les aspects suivants : -La définition des Arts technologiques et l’IA comme outil pour les Artistes. -Le développement d’une Théorie appliquée des Arts Numériques ou technologiques sous l’Angle de
la SAT et de la publication Rencontres entre Artistes et Ingénieurs autour du Numérique (Sous l

--- Résultat 3 (score : 0.653) ---
notions d’Arts technologiques et d’Arts numériques. -L’élaboration d’une Épistémologie de l’Art numérique en général et de la formalisation au pluriel de ce concept sous l’appellation récente « A